# SafeRoad AI — YOLOv8 Pothole Training
### Dataset: RDD2022 India Pothole D40 (84 MB) → ONNX

**Hardware:** GPU T4 x2 (~15–20 minutes training time)
**Output:** `saferoad_model_v2.zip` containing `pothole_yolov8.onnx` and `pothole_yolov8.pt`

## Cell 1: Install & Verify GPU

In [ ]:
!pip install ultralytics kagglehub --quiet

import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
else:
    print('⚠️ Please turn on GPU: Settings (right sidebar) -> Accelerator -> GPU T4 x2')

## Cell 2: Download RDD2022 India & Prepare YOLO Format

In [ ]:
import os, shutil, random, xml.etree.ElementTree as ET
from pathlib import Path
import kagglehub

print('Downloading RDD2022 India Pothole D40 dataset (84 MB)...')
dataset_path = Path(kagglehub.dataset_download('vidishbijalwan/rdd2022-india-pothole-d40'))
print('✅ Downloaded to:', dataset_path)

# Setup output directory
YOLO_DIR = Path('/kaggle/working/yolo_dataset')
for split in ['train', 'val']:
    (YOLO_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)

# Helper: Parse VOC XML
def parse_xml(xml_file):
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        W = int(root.findtext('size/width', '0'))
        H = int(root.findtext('size/height', '0'))
        if W <= 0 or H <= 0: return []
        lines = []
        for obj in root.findall('object'):
            bb = obj.find('bndbox')
            if bb is None: continue
            xmin = max(0.0, float(bb.findtext('xmin', '0')))
            ymin = max(0.0, float(bb.findtext('ymin', '0')))
            xmax = min(float(W), float(bb.findtext('xmax', str(W))))
            ymax = min(float(H), float(bb.findtext('ymax', str(H))))
            if xmax <= xmin or ymax <= ymin: continue
            cx = ((xmin + xmax) / 2) / W
            cy = ((ymin + ymax) / 2) / H
            bw = (xmax - xmin) / W
            bh = (ymax - ymin) / H
            lines.append(f'0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
        return lines
    except:
        return []

# Find all images and matching xmls
images = list(dataset_path.rglob('*.jpg')) + list(dataset_path.rglob('*.png'))
xml_map = {x.stem: x for x in dataset_path.rglob('*.xml')}

pairs = []
for img in images:
    xml = xml_map.get(img.stem)
    if xml:
        boxes = parse_xml(xml)
        if boxes:
            pairs.append((img, boxes))

print(f'✅ Found {len(pairs)} images with annotated potholes!')

# Train/Val 85/15 split
random.seed(42)
random.shuffle(pairs)
n_val = max(1, int(len(pairs) * 0.15))
train_pairs = pairs[n_val:]
val_pairs   = pairs[:n_val]

for split, items in [('train', train_pairs), ('val', val_pairs)]:
    for img, boxes in items:
        shutil.copy2(img, YOLO_DIR / split / 'images' / img.name)
        (YOLO_DIR / split / 'labels' / f'{img.stem}.txt').write_text('\n'.join(boxes))

# Generate data.yaml
DATA_YAML = YOLO_DIR / 'data.yaml'
DATA_YAML.write_text(
    f'path: {YOLO_DIR.as_posix()}\n'
    f'train: train/images\n'
    f'val: val/images\n'
    f'nc: 1\n'
    f"names: ['Pothole']\n"
)
print(f'✅ Data config ready: {DATA_YAML}')
print(f'   Train images: {len(train_pairs)} | Val images: {len(val_pairs)}')

## Cell 3: Train YOLOv8n, Export ONNX & Create Download ZIP

In [ ]:
from ultralytics import YOLO
import zipfile
from pathlib import Path

# 1. Train on GPU (40 epochs takes ~15 mins)
model = YOLO('yolov8n.pt')

results = model.train(
    data=str(DATA_YAML),
    epochs=40,
    imgsz=640,
    batch=32,
    device=0 if torch.cuda.is_available() else 'cpu',
    project='/kaggle/working/runs',
    name='rdd2022_india',
    exist_ok=True,
    patience=10,
    # Indian road augmentations (sun glare, shadows)
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    mosaic=1.0,
)

# 2. Export ONNX (compatible with CPU in SafeRoad app)
best_pt = Path('/kaggle/working/runs/rdd2022_india/weights/best.pt')
print(f'\n✅ Best model checkpoint: {best_pt}')

best_model = YOLO(str(best_pt))
onnx_file = best_model.export(format='onnx', imgsz=640, opset=12, simplify=True)

# 3. Create download zip in Output tab
output_zip = Path('/kaggle/working/saferoad_model_v2.zip')
with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(onnx_file, 'pothole_yolov8.onnx')
    zf.write(best_pt,   'pothole_yolov8.pt')

print('\n' + '=' * 60)
print('🎉 TRAINING & ONNX EXPORT COMPLETE!')
print('👉 Look at the Output panel on the right sidebar.')
print('👉 Click the 3 dots next to saferoad_model_v2.zip and click Download.')
print('=' * 60)